[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/Advanced_Vector_Store_and_Search.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 高级向量存储——轻松上手（Advanced Vector Store - Made Easy）

## 你将学到什么

本 notebook 展示真实应用里使用向量库的**实用方法**。每个示例都简单、开箱即用。

### 先看主线

13 篇搭好了图书馆（存、搜、滤、管四段），这一篇是**实战手册**：四个日常高频问题各配一节——选什么索引、怎么精确过滤、怎么合并多路结果、怎么隔离多用户数据——最后附速查表。备好嵌入（Part 0）后逐个练。

### 主题

1. **选对索引** —— 什么时候用哪个
2. **智能过滤** —— 精确找到你要的
3. **合并结果** —— 融合来自不同来源的搜索
4. **组织数据** —— 不同用户的数据分开存放

---

In [ ]:
!pip install semantica


## Part 0：配置嵌入

先选嵌入提供商和模型。Semantica 支持 Sentence Transformers、FastEmbed 等多家。

In [ ]:
from semantica.embeddings import TextEmbedder

# Choose provider and model
embedder = TextEmbedder(method="fastembed", model_name="BAAI/bge-small-en-v1.5")
dimension = embedder.get_embedding_dimension()

print(f"Selected model: {embedder.get_model_info()['model_name']}")
print(f"Embedding dimension: {dimension}")


## Part 1：选对索引

把索引想成选归档系统：
- **Flat**：像小笔记本——慢但完美（精确）
- **HNSW**：像管理良好的图书馆——快且准
- **IVF**：像分区的仓库——超大集合时飞快

### 简单规则
- 不到 1 万条？用 **Flat**
- 1 万到 100 万？用 **HNSW** ✅（推荐）
- 超过 100 万？用 **IVF**

In [ ]:
from semantica.vector_store import FAISSStore
import numpy as np

# Create some example vectors (like document embeddings)
vectors = np.random.rand(5000, 768).astype('float32')
query = np.random.rand(768).astype('float32')

adapter = FAISSStore(dimension=768)

# HNSW Index - Best for most cases
index = adapter.create_index(index_type="hnsw", metric="L2", m=16)
adapter.add_vectors(vectors, ids=[f"doc_{i}" for i in range(len(vectors))])

# Search for similar vectors
results = adapter.search_similar(query, k=5)

print("Found 5 most similar documents:")
for i, result in enumerate(results, 1):
    print(f"  {i}. Document {result['id']} (distance: {result['distance']:.3f})")

## Part 2：带元数据的智能过滤

想象搜"相似文章"，但只要 2024 年的、只要"科技"分类的——这就是元数据过滤干的事（13 篇的题眼，这里手把手练）。

### 真实场景
你在建文档搜索，用户要：
- 相似的文档（向量搜索）
- 特定分类（元数据过滤）
- 近年（元数据过滤）

In [ ]:
from semantica.vector_store import HybridSearch, MetadataFilter
import numpy as np

# Create sample documents with metadata
documents = [
    {"id": 0, "text": "AI in Healthcare", "category": "Technology", "year": 2024},
    {"id": 1, "text": "Machine Learning Basics", "category": "Technology", "year": 2023},
    {"id": 2, "text": "Business Strategy", "category": "Business", "year": 2024},
    {"id": 3, "text": "Data Science Guide", "category": "Technology", "year": 2024},
    {"id": 4, "text": "Marketing Tips", "category": "Business", "year": 2023},
]

# Create vectors for each document
vectors = [np.random.rand(768) for _ in documents]
metadata = [{"category": d["category"], "year": d["year"]} for d in documents]
vector_ids = [f"doc_{d['id']}" for d in documents]

# Create search
search = HybridSearch()
query = np.random.rand(768)

# Example 1: Find Technology articles from 2024
filter1 = MetadataFilter().eq("category", "Technology").eq("year", 2024)
results = search.search(query, vectors, metadata, vector_ids, filter=filter1, k=10)

print("Technology articles from 2024:")
for r in results:
    doc_id = int(r['id'].split('_')[1])
    print(f"  - {documents[doc_id]['text']}")

# Example 2: Find any article from 2024
filter2 = MetadataFilter().eq("year", 2024)
results2 = search.search(query, vectors, metadata, vector_ids, filter=filter2, k=10)

print("\nAll articles from 2024:")
for r in results2:
    doc_id = int(r['id'].split('_')[1])
    print(f"  - {documents[doc_id]['text']} ({documents[doc_id]['category']})")

## Part 3：合并搜索结果

有时你要在多处搜索再合并结果——比如同时搜邮件和文档，把两边最好的匹配一起展示（13 篇的 RRF，两种口味实操）。

### 什么时候用
- 搜多个数据库
- 组合不同搜索策略
- 给某些来源更高权重

In [ ]:
from semantica.vector_store import SearchRanker

# Simulate two different searches
# Search 1: Recent documents
recent_results = [
    {"id": "doc_3", "score": 0.95, "source": "recent"},
    {"id": "doc_0", "score": 0.90, "source": "recent"},
    {"id": "doc_2", "score": 0.85, "source": "recent"},
]

# Search 2: Popular documents
popular_results = [
    {"id": "doc_1", "score": 0.92, "source": "popular"},
    {"id": "doc_3", "score": 0.88, "source": "popular"},
    {"id": "doc_4", "score": 0.80, "source": "popular"},
]

# Method 1: Fair combination (RRF)
ranker = SearchRanker(strategy="reciprocal_rank_fusion")
combined = ranker.rank([recent_results, popular_results])

print("Combined results (fair ranking):")
for i, result in enumerate(combined[:3], 1):
    doc_id = int(result['id'].split('_')[1])
    print(f"  {i}. {documents[doc_id]['text']} (score: {result['score']:.3f})")

# Method 2: Prefer recent documents (70% recent, 30% popular)
weighted_ranker = SearchRanker(strategy="weighted_average")
weighted_combined = weighted_ranker.rank(
    [recent_results, popular_results],
    weights=[0.7, 0.3]
)

print("\nCombined results (prefer recent):")
for i, result in enumerate(weighted_combined[:3], 1):
    doc_id = int(result['id'].split('_')[1])
    print(f"  {i}. {documents[doc_id]['text']} (score: {result['score']:.3f})")

## Part 4：用户数据分开存放

做多人或多公司的应用，数据必须隔离——命名空间自动办妥（13 篇的"分柜"，SaaS 实战版）。

### 真实例子
你在建 SaaS 应用：
- A 公司有 A 公司的文档
- B 公司有 B 公司的文档
- 两家永远不该看到对方的数据

In [ ]:
from semantica.vector_store import NamespaceManager

# Create manager
manager = NamespaceManager()

# Create separate spaces for each company
company_a = manager.create_namespace("company_a", "Company A's documents")
company_b = manager.create_namespace("company_b", "Company B's documents")

# Add documents to Company A
for i in range(10):
    manager.add_vector_to_namespace(f"company_a_doc_{i}", "company_a")

# Add documents to Company B
for i in range(15):
    manager.add_vector_to_namespace(f"company_b_doc_{i}", "company_b")

# Get each company's documents
a_docs = manager.get_namespace_vectors("company_a")
b_docs = manager.get_namespace_vectors("company_b")

print(f"Company A has {len(a_docs)} documents")
print(f"Company B has {len(b_docs)} documents")

# Set permissions (who can access what)
company_a.set_access_control("admin@companya.com", ["read", "write", "delete"])
company_a.set_access_control("user@companya.com", ["read"])  # Read-only

# Check permissions
print(f"\nAdmin can delete: {company_a.has_permission('admin@companya.com', 'delete')}")
print(f"User can delete: {company_a.has_permission('user@companya.com', 'delete')}")

## 速查指南

### 该用哪个索引？

```python
# 小数据集（< 10,000 条）
index = adapter.create_index(index_type="flat", metric="L2")

# 中数据集（10,000 - 1,000,000 条）✅ 推荐
index = adapter.create_index(index_type="hnsw", metric="L2", m=16)

# 大数据集（> 1,000,000 条）
index = adapter.create_index(index_type="ivf", metric="L2", nlist=100)
```

### 怎么过滤结果？

```python
# 单条件
filter = MetadataFilter().eq("category", "Technology")

# 多条件（AND）
filter = MetadataFilter() \
    .eq("category", "Technology") \
    .eq("year", 2024)

# 大于 / 小于
filter = MetadataFilter().gt("year", 2020)
```

### 怎么合并结果？

```python
# 公平合并
ranker = SearchRanker(strategy="reciprocal_rank_fusion")
combined = ranker.rank([results1, results2])

# 加权合并（偏向第一个来源）
ranker = SearchRanker(strategy="weighted_average")
combined = ranker.rank([results1, results2], weights=[0.7, 0.3])
```

### 怎么隔离用户数据？

```python
# 给每个用户/公司建命名空间
manager = NamespaceManager()
user_space = manager.create_namespace("user_123", "User 123's data")

# 往命名空间加数据
manager.add_vector_to_namespace("doc_1", "user_123")

# 取用户的数据
user_docs = manager.get_namespace_vectors("user_123")
```

---

## 小结

### 回顾主线

四件实用技能到手：**选索引**（按数据量：Flat/HNSW/IVF）、**智能过滤**（向量相似 + 元数据双条件）、**结果融合**（RRF 公平合并或加权偏置）、**数据隔离**（命名空间分柜 + 权限）。13 篇的图书馆，这一篇教你当管理员。

你已学会：

1. ✅ **索引选择**：多数场景用 HNSW
2. ✅ **智能过滤**：向量搜索结合元数据
3. ✅ **结果融合**：合并不同来源的搜索
4. ✅ **数据隔离**：用户数据分开存放

### 下一步

- 用自己的数据试试这些示例
- 实验不同的过滤条件
- 构建多用户应用
- 更多基础见[入门 notebook](../introduction/13_Vector_Store.ipynb)

**需要帮助？** 查我们的[文档](https://semantica.readthedocs.io)或在 [GitHub](https://github.com/semantica-agi/semantica) 提问。